# 06 - Analisis biaya retrieval dan ekspor hasil

Tiga hal: mengukur biaya indeks FAISS yang menjadi satu-satunya biaya tambahan
RM-c, menggabungkan riwayat run bila ada lebih dari satu folder kampanye, dan
mengekspor seluruh artefak ke satu berkas Excel untuk penulisan Bab 4.

In [1]:
import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir
runner = CampaignRunner(out_dir=OUT_DIR)
features = runner.features
print("fitur beku:", {k: v.shape for k, v in features.embeddings.items()})

/workspace/indobert-with-rac/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-14 17:39:36,092 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-base-p2


2026-09-14 17:39:37,413 | INFO     | src.services.features | Fitur beku dimuat dari cache /workspace/indobert-with-rac/outputs/tuning/features/indobenchmark__indobert-base-p2
fitur beku: {'train': (6588, 768), 'val': (1402, 768), 'test': (1405, 768)}


## 1. Biaya indeks FAISS

RM-c tidak melatih apa pun (biaya latihnya adalah biaya head RM-b yang dipakainya),
tetapi menambah biaya retrieval saat inferensi. Biaya itu baru jujur bila ikut
diukur, dan biayanya terbagi dua: pembangunan indeks sekali dari embedding train,
serta penelusuran pada setiap inferensi yang tumbuh mengikuti k.

In [2]:
from src.services.faiss_benchmark import FaissBenchmark

benchmark = FaissBenchmark(
    features.embeddings["train"], features.labels["train"], repeats=5
)
hasil = benchmark.run(
    features.embeddings["test"],
    k_values=(1, 3, 5, 10, 20, 50),
    out_dir=settings.output_dir / "faiss_index",
)

print(pd.Series(hasil.build).to_string())
pd.DataFrame(hasil.search)

2026-09-14 17:39:37,571 | INFO     | src.services.faiss_benchmark | Bangun indeks: 6588 vektor x 768 dim dalam 4.01 ms (19.30 MB)
2026-09-14 17:39:38,259 | INFO     | src.services.faiss_benchmark | Telusur k=1: 129.480 ms untuk 1405 query (92.16 us/query)
2026-09-14 17:39:39,852 | INFO     | src.services.faiss_benchmark | Telusur k=3: 318.115 ms untuk 1405 query (226.42 us/query)
2026-09-14 17:39:41,459 | INFO     | src.services.faiss_benchmark | Telusur k=5: 320.907 ms untuk 1405 query (228.40 us/query)
2026-09-14 17:39:43,188 | INFO     | src.services.faiss_benchmark | Telusur k=10: 345.380 ms untuk 1405 query (245.82 us/query)
2026-09-14 17:39:44,857 | INFO     | src.services.faiss_benchmark | Telusur k=20: 333.393 ms untuk 1405 query (237.29 us/query)
2026-09-14 17:39:46,546 | INFO     | src.services.faiss_benchmark | Telusur k=50: 337.348 ms untuk 1405 query (240.11 us/query)
2026-09-14 17:39:46,575 | INFO     | src.services.faiss_benchmark | Hasil benchmark FAISS ditulis ke /work

,k,n_queries,search_time_ms_batch,search_time_us_per_query
0,1,1405,129.4799,92.1565
1,3,1405,318.1149,226.4163
2,5,1405,320.9070,228.4036
3,10,1405,345.3798,245.8219
4,20,1405,333.3932,237.2906
5,50,1405,337.3479,240.1053


Indeks bertipe flat/exact, jadi penelusuran adalah brute force atas seluruh
vektor train. Untuk ukuran data ini biayanya sepele dan hasilnya deterministik,
yang jauh lebih penting untuk penelitian daripada penghematan waktu dari indeks
aproksimasi.

In [7]:
import matplotlib.pyplot as plt

search = pd.DataFrame(hasil.search)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(search["k"], search["search_time_us_per_query"], marker="o")
ax.set_xlabel("k (jumlah tetangga)")
ax.set_ylabel("mikrodetik per query")
ax.set_title("Biaya penelusuran indeks FAISS")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Gabungkan riwayat lintas folder kampanye

Hanya perlu bila ada lebih dari satu folder keluaran, misalnya kampanye utama
ditambah eksplorasi dengan encoder berbeda.

In [4]:
from src.services.aggregation import RunMerger

SUMBER = {"tuning": OUT_DIR}

if len(SUMBER) > 1:
    ditulis = RunMerger(SUMBER).merge_all(settings.output_dir / "combined")
    for kunci, path in ditulis.items():
        print(f"  {kunci}: {path}")
else:
    print("hanya satu folder kampanye; penggabungan dilewati")

hanya satu folder kampanye; penggabungan dilewati


Saat membaca hasil gabungan, kunci barisnya adalah pasangan (`source`, `run_id`)
karena tiap folder memulai penomoran dari 1. Kolom waktu, memori, dan latency
tidak boleh dibandingkan lintas `source`.

## 3. Ekspor ke Excel

In [5]:
from src.services.workbook import WorkbookBuilder

builder = WorkbookBuilder(OUT_DIR)
path = builder.build(settings.data_dir.parent / "HASIL.xlsx")
print(f"{path} ({path.stat().st_size / 1024:.0f} KB)")

for nama, frame in builder.sheets().items():
    print(f"  {nama:28s}: {len(frame):4,} baris x {len(frame.columns)} kolom")

2026-09-14 17:39:46,850 | INFO     | src.services.workbook | Workbook ditulis ke /workspace/indobert-with-rac/HASIL.xlsx (14 sheet)
/workspace/indobert-with-rac/HASIL.xlsx (59 KB)
  Ringkasan                   :   27 baris x 2 kolom
  Run RMA                     :   27 baris x 27 kolom
  Kurva RMA                   :  143 baris x 8 kolom
  Run RMB                     :   27 baris x 29 kolom
  Kurva RMB                   :  345 baris x 8 kolom
  Run RMC                     :   67 baris x 21 kolom
  Perbandingan Final          :    3 baris x 12 kolom
  Kriteria Sukses             :    2 baris x 11 kolom
  Benchmark Inferensi         :    3 baris x 3 kolom
  Pivot rma_grid_pivot_batch16:    4 baris x 4 kolom
  Pivot rma_grid_pivot_batch32:    4 baris x 4 kolom
  Pivot rmb_grid_pivot_hidden_dim:    1 baris x 2 kolom
  Pivot rmc_grid_pivot_weightings:   11 baris x 7 kolom
  Pivot rmc_grid_pivot_weightingu:    1 baris x 2 kolom


## 4. Regenerasi seluruh figur

In [6]:
from src.services.reporting import FigureReporter

reporter = FigureReporter(OUT_DIR)
for skenario in ("rma", "rmb", "rmc"):
    dibuat = reporter.refresh_scenario(skenario)
    print(f"{skenario}: {len(dibuat)} artefak")

reporter.final_inference_bar_chart()
reporter.write_summary()
print(f"\ntotal figur: {len(list((OUT_DIR / 'figures').glob('*.png')))}")

rma: 6 artefak
rmb: 12 artefak
rmc: 6 artefak

total figur: 80


## Ringkasan

Bahan Bab 4 lengkap: `HASIL.xlsx` di root, tabel metrik di
`outputs/tuning/metrics/`, figur di `outputs/tuning/figures/`, dan biaya
retrieval di `outputs/faiss_index/`.